# Day 09. Exercise 00
# Regularization

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
import joblib

ModuleNotFoundError: No module named 'seaborn'

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [ ]:
df = pd.read_csv('../data/dayofweek.csv')

df.head()

In [ ]:
target = 'dayofweek'
feature = [c for c in df.columns if c != target]

X = df[feature]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [ ]:
def crossval(n_splits, X, y, model, print_result=True):
    skf = StratifiedKFold(n_splits=n_splits)
    accuracy_scores = []
    for i, (train_index, test_index) in enumerate(skf.split(X, y)):
        X_train_crossval = X.iloc[train_index, :]
        X_test_crossval = X.iloc[test_index, :]

        model.fit(X_train_crossval, y[train_index])

        y_train_predict = model.predict(X_train_crossval)
        y_test_predict = model.predict(X_test_crossval)

        accuracy_train = accuracy_score(y[train_index], y_train_predict)
        accuracy_valid = accuracy_score(y[test_index], y_test_predict)
        accuracy_scores.append(accuracy_valid)
        if print_result:
            print(f'train - {accuracy_train:.5f}  |  valid - {accuracy_valid:.5f}')

    if print_result:
        print(f'Average accuracy on crossval is {np.mean(accuracy_scores):.5f}')
        print(f'Std is {np.std(accuracy_scores):.5f}')

In [ ]:
%%time
model = LogisticRegression(random_state=21, fit_intercept=False)
crossval(10, X_train, y_train, model)

### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [ ]:
%%time
model = LogisticRegression(penalty='none', random_state=21, fit_intercept=False, solver='newton-cg')
crossval(10, X_train, y_train, model)

In [ ]:
%%time
model = LogisticRegression(penalty='l1', random_state=21, fit_intercept=False, solver='liblinear')
crossval(10, X_train, y_train, model)

In [ ]:
%%time
model = LogisticRegression(penalty='l2', random_state=21, fit_intercept=False, solver='newton-cg')
crossval(10, X_train, y_train, model)

## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [ ]:
%%time
model = SVC(probability=True, kernel='linear', random_state=21)
crossval(10, X_train, y_train, model)

### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [ ]:
Cs = np.logspace(-1, 2, 4)

In [ ]:
%%time
model = SVC(C=Cs[0], probability=True, kernel='linear', random_state=21)
crossval(10, X_train, y_train, model)

In [ ]:
%%time
model = SVC(C=Cs[2], probability=True, kernel='linear', random_state=21)
crossval(10, X_train, y_train, model)

In [ ]:
%%time
model = SVC(C=Cs[3], probability=True, kernel='linear', random_state=21)
crossval(10, X_train, y_train, model)

## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [ ]:
%%time
model = DecisionTreeClassifier(max_depth=10, random_state=21)
crossval(10, X_train, y_train, model)

### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [ ]:
%%time
model = DecisionTreeClassifier(max_depth=25, random_state=21)
crossval(10, X_train, y_train, model)

In [ ]:
%%time
model = DecisionTreeClassifier(max_depth=4, random_state=21)
crossval(10, X_train, y_train, model)

In [ ]:
%%time
model = DecisionTreeClassifier(max_depth=15, random_state=21)
crossval(10, X_train, y_train, model)

In [ ]:
%%time
model = DecisionTreeClassifier(max_depth=25, random_state=21, min_samples_split=2, min_samples_leaf=1)
crossval(10, X_train, y_train, model)

## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [ ]:
%%time
model = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
crossval(10, X_train, y_train, model)

### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [ ]:
%%time
model = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=21)
crossval(10, X_train, y_train, model)

In [ ]:
%%time
model = RandomForestClassifier(n_estimators=100, max_depth=25, random_state=21)
crossval(10, X_train, y_train, model)

In [ ]:
%%time
model = RandomForestClassifier(n_estimators=50, max_depth=20, random_state=21)
crossval(10, X_train, y_train, model)

In [ ]:
%%time
model = RandomForestClassifier(n_estimators=100, max_depth=25, random_state=21, min_samples_split=4, min_samples_leaf=2)
crossval(10, X_train, y_train, model)

In [ ]:
%%time
model = RandomForestClassifier(n_estimators=150, max_depth=25, random_state=21, min_samples_split=4, min_samples_leaf=2)
crossval(10, X_train, y_train, model)

## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [ ]:
%%time
model = RandomForestClassifier(n_estimators=100, max_depth=25, random_state=21)
model.fit(X_train, y_train)

In [ ]:
%%time
model.score(X_test, y_test)

In [ ]:
y_pred = model.predict(X_test)

df = pd.DataFrame({'actual': y_test})
df['is_error'] = y_test != y_pred

result = df.groupby('actual').is_error.agg(
    count='size',
    errors_count='sum'
)

result['error %'] = np.round(result['errors_count'] / result['count'] * 100, 2)

In [ ]:
result

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True)

plt.xlabel('Predicted')
plt.ylabel('True')

plt.show()

In [ ]:
joblib.dump(model, '../data/rfc_model.joblib')